# Setup

In [0]:
%pip install -r ../requirements.txt

In [0]:
import re
from _common import run_script

In [0]:
_current_user = spark.sql("SELECT current_user()").collect()[0][0]
_nickname     = re.sub(r"[^a-z0-9]", "_", _current_user.split("@")[0].lower())
print(f"Username  : {_nickname}")

In [0]:
catalog  = "demo_30_giugno"
n_orders = 50_000

# Generazione dati sintetici

In [0]:
run_script("01_generate_data.py", catalog=catalog, schema=_nickname, n_orders=n_orders, warehouse_id="2679e635f440f0c6")

In [0]:
run_script("02_generate_ka_documents.py", catalog=catalog, schema=_nickname)

# Creazione Knowledge Assistant
- **KA Name:**
  **{your_nickname}**_KA
- **KA Description:**
  RAG sui documenti interni di Panino Bricks.
- **Knowledge source description:**
  6 PDF: manuale operatore, ricettario, HACCP/allergeni, franchising, manutenzione, fornitori.
- **Knowledge source path:**
  /Volumes/demo_30_giugno/**{your_nickname}**/ka_documents/
- **Knowledge source name:**
  Documenti Interni Panino Bricks
- **Instructions:**
  Sei l'assistente operativo di Panino Bricks, catena italiana di paninoteche. 
  Rispondi SEMPRE in italiano, basandoti esclusivamente sui documenti interni forniti
  (manuale operatore, ricettario panini, politica di sicurezza alimentare HACCP e allergeni,
  guida franchising, manutenzione attrezzature, accordi fornitori).
  Per domande su allergeni o idoneita' a diete particolari (es. clienti celiaci), cita sempre
  la politica di sicurezza alimentare e ricorda che i panini contengono glutine e che non c'e'
  garanzia gluten-free per contaminazione.
  Se l'informazione non e' presente nei documenti, dillo chiaramente invece di inventare.

# Creazione Supervisor
- **Supervisor Name:**
  **{your_nickname}**_supervisor
- **Linked Genie Space ID:**
  **{your_geniespace_id}**
- **Linked KA Name:**
  **{your_nickname}**_KA
- **Supervisor Instructions:**
  Sei il supervisore di Panino Bricks HQ, catena italiana di paninoteche.
  Rispondi SEMPRE in italiano. Instrada ogni richiesta al sottoagente giusto:
  usa 'Analista Vendite' (Genie) per domande su dati strutturati (vendite, scontrini, panini piu' venduti,
  scorte/magazzino, clienti, fedelta', promozioni, fatturato, performance per punto vendita);
  usa 'Ops Bot' (Knowledge Assistant) per ricette dei panini, procedure operative, sicurezza alimentare e
  allergeni (HACCP), franchising, manutenzione attrezzature e accordi con i fornitori.
  Per domande composte (che richiedono sia dati sia procedure) interpella entrambi i sottoagenti e sintetizza una risposta unica.
- **Supervisor Description:**
  Supervisore multi-agente Panino Bricks: instrada tra Genie (vendite/ops) e Knowledge Assistant (documenti).

# Creazione Genie Space

In [0]:
_out03 = run_script("03_create_genie_space.py", catalog=catalog, schema=_nickname, warehouse_id="2679e635f440f0c6")

# Grant Permessi al SP

In [0]:
sp = "YOUR-APP-SP-ID"
genie_space_id = "YOUR-GENIE-SPACE-ID"
mas_endpoint = "YOUR-MAS-ENDPOINT-NAME"
ka_endpoint = "YOUR-KA-ENDPOINT-NAME"

In [0]:
run_script("06_grant_app_permissions.py",
           catalog=catalog,
           schema=_nickname,
           sp=sp,
           genie_space_id=genie_space_id,
           mas_endpoint=mas_endpoint,
           ka_endpoint=ka_endpoint,
           warehouse_id="2679e635f440f0c6")

In [0]:
run_script("07_grant_lakebase_role.py", sp=sp)